In [ ]:
import numpy as np
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
import json

In [2]:
notes = pd.read_csv("data/synthetic_ntds_trauma_notes_gemini.csv", index_col = "encounter_id")

In [58]:
with open("./data/ntds_18_complications.json", "r") as f:
    complications_info = json.load(f)
complications_info

[{'id': 'aki',
  'label': 'Acute Kidney Injury',
  'short_definition': 'New kidney dysfunction during this hospitalization, such as a rise in serum creatinine or oliguria/anuria, not clearly present before admission.',
  'positive_note_clues': ['developed acute kidney injury with rising creatinine',
   'oliguria requiring nephrology consultation',
   'initiated dialysis for new renal failure']},
 {'id': 'aws',
  'label': 'Alcohol Withdrawal Syndrome',
  'short_definition': 'Clinical alcohol withdrawal that began after admission, with symptoms such as tremor, agitation, hallucinations, or withdrawal seizures.',
  'positive_note_clues': ['placed on alcohol withdrawal protocol with high CIWA scores',
   'developed agitation and tremors consistent with alcohol withdrawal',
   'treated with benzodiazepines for withdrawal symptoms']},
 {'id': 'ards',
  'label': 'Acute Respiratory Distress Syndrome',
  'short_definition': 'Acute hypoxemic respiratory failure with bilateral lung infiltrates no

In [ ]:
def format_complications_info(complications_info):
    output_lines = []
    
    for comp in complications_info:
        # Add label and short_definition
        output_lines.append(f"{comp['label']}: {comp['short_definition']}")
        
        output_lines.append("Positive Note Clues:")
        for i, clue in enumerate(comp['positive_note_clues'], 1):
            output_lines.append(f"{i}. {clue}")
        
        # Add blank line between complications
        output_lines.append("")
    
    return "\n".join(output_lines)

# Test the function
complications_str = format_complications_info(complications_info)
print(complications_str)

Acute Kidney Injury: New kidney dysfunction during this hospitalization, such as a rise in serum creatinine or oliguria/anuria, not clearly present before admission.
Positive Note Clues:
1. developed acute kidney injury with rising creatinine
2. oliguria requiring nephrology consultation
3. initiated dialysis for new renal failure

Alcohol Withdrawal Syndrome: Clinical alcohol withdrawal that began after admission, with symptoms such as tremor, agitation, hallucinations, or withdrawal seizures.
Positive Note Clues:
1. placed on alcohol withdrawal protocol with high CIWA scores
2. developed agitation and tremors consistent with alcohol withdrawal
3. treated with benzodiazepines for withdrawal symptoms

Acute Respiratory Distress Syndrome: Acute hypoxemic respiratory failure with bilateral lung infiltrates not fully explained by cardiac failure or fluid overload, meeting ARDS criteria during the stay.
Positive Note Clues:
1. worsening hypoxemia with bilateral infiltrates consistent with 

In [62]:
# I can feed in the actual possible conditions as a system prompt. Then I can feed in the
# notes as a user prompt 
test_template = ChatPromptTemplate(
    [
    ("system", """You are an expert registrar who is highly experienced at meeting the
    National Trauma Data Standard (NTDS). You will be provided a patient's medical note
    from UCSD Health, a level 1 health center. Please determine which of the conditions the 
    patient has. Below are the possible complications:""" + '\n' + complications_str),
    ("human", "{note}"),
    ("system", """The following chunks were to be relevant to the final outputs.
     Please output your answers as a JSON with a 1 indicating the patient has the disorder
     and a zero indicating that the patient does not. Include all 18 fields.""")
    ]
)

In [66]:

sample_note = notes.loc[0, 'note_text']
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, 
                                          chunk_overlap = 200, 
                                          separators = ["\n\n", "\n", " ", ""])
chunked = splitter.split_text(sample_note)
chunked

['**ED TRAUMA H&P:**\nThis 79-year-old male was brought to the trauma bay via EMS after a significant fall from a height of approximately 15 feet while working on his roof. Per EMS report, he was found alert but confused at the scene by family and complained of diffuse body pain. On arrival, initial vital signs were heart rate 97 bpm, blood pressure 120/78 mmHg (MAP 92 mmHg), respiratory rate 22 breaths/min, and oxygen saturation 95% on a 4L nasal cannula, later titrated to 6L to maintain saturation >94%.',
 'Primary survey was completed rapidly per ATLS protocol. Airway was patent and protected. Breath sounds were clear bilaterally, though shallow, with no obvious respiratory distress. Cardiovascularly, peripheral pulses were palpable and strong, skin was warm and dry, capillary refill brisk. Neurologically, he presented with a GCS of 14 (E4V4M6), oriented to person but confused to place and time, without obvious focal deficits upon initial assessment. Gross deformities were noted to 

In [ ]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os

# Make sure GOOGLE_API_KEY is set)
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

# Initialize ChromaDB with persistence
vectorstore = Chroma(
    collection_name="ntds_notes",
    embedding_function=embeddings,
    persist_directory="./ntds_embeddings"
)

# Initialize text splitter with same parameters as before
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200, 
    separators=["\n\n", "\n", " ", ""]
)

# Process all notes
all_chunks = []
all_metadatas = []

print(f"Processing {len(notes)} notes...")
for idx, (encounter_id, row) in enumerate(notes.iterrows()):
    note_text = row['note_text']
    
    # Chunk the note
    chunks = splitter.split_text(note_text)
    
    # Create metadata for each chunk
    for chunk_idx, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_metadatas.append({
            'encounter_id': str(encounter_id),
            'chunk_index': chunk_idx,
            'total_chunks': len(chunks)
        })
    
    if (idx + 1) % 10 == 0:
        print(f"Processed {idx + 1}/{len(notes)} notes...")

# Add all chunks to ChromaDB
print(f"\nAdding {len(all_chunks)} chunks to ChromaDB...")
vectorstore.add_texts(texts=all_chunks, metadatas=all_metadatas)

print(f"Successfully created ChromaDB with {len(all_chunks)} chunks")
print(f"Persisted to: ./ntds_embeddings")

/opt/miniconda3/envs/ntds-extractor/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.14) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


DefaultCredentialsError: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.